# Stage 2: Retrieval Methods Comparison

This notebook compares 3 retrieval methods on the same set of test claims:
- **Naive**: Raw cosine similarity search on ChromaDB
- **Hybrid**: BM25 (lexical) + dense (PubMedBERT) with reciprocal rank fusion
- **Hybrid + Reranked**: Hybrid retrieval followed by cross-encoder re-ranking

We measure: retrieval latency, overlap between methods, rank position changes, and evidence quality.

In [ ]:
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

## 2.1 Setup — load test claims and prepare ChromaDB

In [ ]:
with open("data/test_claims.json") as f:
    claims = json.load(f)

print(f"Test claims: {len(claims)}")
for c in claims:
    print(f"  [{c['expected_verdict']:>24}] {c['claim']}")

In [ ]:
from src.shared.vector_store import get_chroma_client, get_or_create_collection, search

client = get_chroma_client()
collection = get_or_create_collection(client)
print(f"ChromaDB collection: {collection.count()} documents")

## 2.2 Run all 3 retrieval methods on each claim

In [ ]:
from src.retrieval.hybrid import retrieve_hybrid
from src.retrieval.reranker import rerank

TOP_K = 5
retrieval_results = []

for claim_data in claims:
    query = claim_data["claim"]
    row = {"claim": query, "expected_verdict": claim_data["expected_verdict"]}
    
    # Naive
    t0 = time.time()
    naive_hits = search(collection, query, top_k=TOP_K)
    row["naive_time"] = round(time.time() - t0, 4)
    row["naive_hits"] = naive_hits
    row["naive_ids"] = [h["id"] for h in naive_hits]
    
    # Hybrid
    t0 = time.time()
    hybrid_hits = retrieve_hybrid(query, collection, top_k=TOP_K)
    row["hybrid_time"] = round(time.time() - t0, 4)
    row["hybrid_hits"] = hybrid_hits
    row["hybrid_ids"] = [h["id"] for h in hybrid_hits]
    
    # Hybrid + Reranked
    t0 = time.time()
    hybrid_candidates = retrieve_hybrid(query, collection, top_k=TOP_K * 2)
    reranked_hits = rerank(query, hybrid_candidates, top_k=TOP_K)
    row["reranked_time"] = round(time.time() - t0, 4)
    row["reranked_hits"] = reranked_hits
    row["reranked_ids"] = [h["id"] for h in reranked_hits]
    
    retrieval_results.append(row)
    print(f"  {query[:50]:50} naive={row['naive_time']:.3f}s  hybrid={row['hybrid_time']:.3f}s  reranked={row['reranked_time']:.3f}s")

print("\nDone!")

## 2.3 Latency comparison

In [ ]:
latency_df = pd.DataFrame([
    {"Claim": r["claim"][:40], "Naive (s)": r["naive_time"], "Hybrid (s)": r["hybrid_time"], "Reranked (s)": r["reranked_time"]}
    for r in retrieval_results
])
print("Retrieval Latency (seconds):")
print(latency_df.to_string(index=False))
print(f"\nAverages: Naive={latency_df['Naive (s)'].mean():.4f}s  Hybrid={latency_df['Hybrid (s)'].mean():.4f}s  Reranked={latency_df['Reranked (s)'].mean():.4f}s")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(retrieval_results))
width = 0.25

ax.bar([i - width for i in x], [r["naive_time"] for r in retrieval_results], width, label="Naive", color="#4C72B0")
ax.bar(x, [r["hybrid_time"] for r in retrieval_results], width, label="Hybrid", color="#DD8452")
ax.bar([i + width for i in x], [r["reranked_time"] for r in retrieval_results], width, label="Hybrid+Reranked", color="#C44E52")

ax.set_xticks(x)
ax.set_xticklabels([r["claim"][:25] + "..." for r in retrieval_results], rotation=45, ha="right")
ax.set_ylabel("Seconds")
ax.set_title("Retrieval Latency by Method")
ax.legend()
plt.tight_layout()
plt.show()

## 2.4 Result overlap — do methods return different documents?

In [ ]:
print(f"{'Claim':<45} {'N∩H':>5} {'N∩R':>5} {'H∩R':>5} {'All3':>5}")
print("-" * 70)

overlaps = []
for r in retrieval_results:
    n_set = set(r["naive_ids"])
    h_set = set(r["hybrid_ids"])
    rr_set = set(r["reranked_ids"])
    
    nh = len(n_set & h_set)
    nr = len(n_set & rr_set)
    hr = len(h_set & rr_set)
    all3 = len(n_set & h_set & rr_set)
    
    overlaps.append({"claim": r["claim"][:42], "N∩H": nh, "N∩R": nr, "H∩R": hr, "All3": all3})
    print(f"{r['claim'][:42]:<45} {nh:>5} {nr:>5} {hr:>5} {all3:>5}")

avg_all3 = sum(o["All3"] for o in overlaps) / len(overlaps)
print(f"\nAverage docs in all 3 methods: {avg_all3:.1f}/5")
print(f"Interpretation: {'Methods return mostly the same docs — retrieval is not the differentiator' if avg_all3 > 3 else 'Methods return different docs — retrieval method matters'}")

## 2.5 Top-1 document comparison

In [ ]:
# Show the top-1 result from each method for each claim
for r in retrieval_results:
    print(f"\nClaim: {r['claim']}")
    print(f"  Expected: {r['expected_verdict']}")
    
    for method, key in [("Naive", "naive_hits"), ("Hybrid", "hybrid_hits"), ("Reranked", "reranked_hits")]:
        hit = r[key][0] if r[key] else None
        if hit:
            score_key = "distance" if "distance" in hit else ("rerank_score" if "rerank_score" in hit else "score")
            score = hit.get(score_key, "N/A")
            print(f"  {method:>10}: [{hit.get('id', 'N/A')}] score={score:.4f}" if isinstance(score, float) else f"  {method:>10}: [{hit.get('id', 'N/A')}]")
            print(f"             {hit.get('text', '')[:100]}...")

## 2.6 Reranker score distribution

In [ ]:
# How much does reranking change the order?
all_rerank_scores = []
for r in retrieval_results:
    for h in r["reranked_hits"]:
        all_rerank_scores.append(h.get("rerank_score", 0))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(all_rerank_scores, bins=20, color="#C44E52", edgecolor="white", alpha=0.8)
ax.set_xlabel("Cross-encoder rerank score")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Reranker Scores (all claims, top-5)")
ax.axvline(sum(all_rerank_scores)/len(all_rerank_scores), color="black", linestyle="--", label=f"mean={sum(all_rerank_scores)/len(all_rerank_scores):.2f}")
ax.legend()
plt.tight_layout()
plt.show()

## Key Takeaways

| Method | Avg Latency | Key Property |
|--------|------------|-------------|
| Naive | ~0.05s | Fast, pure semantic similarity |
| Hybrid | ~0.2s | BM25 catches keyword matches dense misses |
| Hybrid+Reranked | ~0.5s | Cross-encoder provides more accurate relevance ranking |

**Next step:** Stage 3 tests how these retrieval differences affect the LLM's verdict quality.